# Notebook goal
I will try unit test, integration test and end-to-end test. I should have integrate this testing process from the beginning (next time I will buil building process with test stage). Also having separate catalog for testing would be mor professional ride_lakehouse_test.

In [0]:
from pyspark.sql import functions as F

CATALOG = "ride_lakehouse_dev"

BRONZE_DRIVERS = f"{CATALOG}.bronze.drivers"
BRONZE_RIDES = f"{CATALOG}.bronze.rides"
BRONZE_PAYMENTS = f"{CATALOG}.bronze.payments"

SILVER_DRIVERS = f"{CATALOG}.silver.drivers"
SILVER_RIDES = f"{CATALOG}.silver.rides"
SILVER_PAYMENTS = f"{CATALOG}.silver.payments"

GOLD_DRIVER_STATS = f"{CATALOG}.gold.driver_stats"
GOLD_DAILY_STATS = f"{CATALOG}.gold.daily_stats"

In [0]:
test_results = []

def check(name, condition, details=""):

    passed = bool(condition)

    status = "PASS" if passed else "FAIL"

    test_results.append({
        "test": name,
        "status": status,
        "details": details
    })

    print(f"[{status}] {name}")

    if details and not passed:
        print(f"        {details}")

# Part 1 — Structural tests

In [0]:
required_tables = [
    BRONZE_DRIVERS,
    BRONZE_RIDES,
    BRONZE_PAYMENTS,

    SILVER_DRIVERS,
    SILVER_RIDES,
    SILVER_PAYMENTS,

    GOLD_DRIVER_STATS,
    GOLD_DAILY_STATS
]

for table in required_tables:

    exists = spark.catalog.tableExists(table)

    check(
        f"Table exists: {table}",
        exists
    )

[PASS] Table exists: ride_lakehouse_dev.bronze.drivers
[PASS] Table exists: ride_lakehouse_dev.bronze.rides
[PASS] Table exists: ride_lakehouse_dev.bronze.payments
[PASS] Table exists: ride_lakehouse_dev.silver.drivers
[PASS] Table exists: ride_lakehouse_dev.silver.rides
[PASS] Table exists: ride_lakehouse_dev.silver.payments
[PASS] Table exists: ride_lakehouse_dev.gold.driver_stats
[PASS] Table exists: ride_lakehouse_dev.gold.daily_stats


## Test schema evolution

In [0]:
bronze_driver_columns = (
    spark.table(BRONZE_DRIVERS).columns
)

silver_driver_columns = (
    spark.table(SILVER_DRIVERS).columns
)

check(
    "Bronze contains phone",
    "phone" in bronze_driver_columns
)

check(
    "Silver contains phone",
    "phone" in silver_driver_columns
)

[PASS] Bronze contains phone
[PASS] Silver contains phone


In [0]:
from pyspark.sql.types import StringType

silver_driver_schema = (
    spark.table(SILVER_DRIVERS).schema
)

if "phone" in silver_driver_columns:

    check(
        "Silver phone is STRING",
        isinstance(
            silver_driver_schema["phone"].dataType,
            StringType
        )
    )

[PASS] Silver phone is STRING


## Verify timestamp types

In [0]:
from pyspark.sql.types import TimestampType

timestamp_columns = {
    SILVER_DRIVERS: [
        "generated_at"
    ],

    SILVER_RIDES: [
        "ride_timestamp",
        "generated_at"
    ],

    SILVER_PAYMENTS: [
        "payment_timestamp",
        "generated_at"
    ]
}

for table, columns in timestamp_columns.items():

    schema = spark.table(table).schema

    for column in columns:

        check(
            f"{table}.{column} is TIMESTAMP",

            isinstance(
                schema[column].dataType,
                TimestampType
            )
        )

[PASS] ride_lakehouse_dev.silver.drivers.generated_at is TIMESTAMP
[PASS] ride_lakehouse_dev.silver.rides.ride_timestamp is TIMESTAMP
[PASS] ride_lakehouse_dev.silver.rides.generated_at is TIMESTAMP
[PASS] ride_lakehouse_dev.silver.payments.payment_timestamp is TIMESTAMP
[PASS] ride_lakehouse_dev.silver.payments.generated_at is TIMESTAMP


# Referential integrity

In [0]:
rides = spark.table(SILVER_RIDES)
drivers = spark.table(SILVER_DRIVERS)
payments = spark.table(SILVER_PAYMENTS)
orphan_rides = (
    rides
    .join(
        drivers,

        rides.driver_id == drivers.id,

        "left_anti"
    )
)

orphan_ride_count = orphan_rides.count()

check(
    "Every ride references a valid driver",
    orphan_ride_count == 0,

    f"Orphan rides: {orphan_ride_count}"
)

[PASS] Every ride references a valid driver


In [0]:
orphan_payments = (
    payments
    .join(
        rides,

        payments.ride_id == rides.ride_id,

        "left_anti"
    )
)

orphan_payment_count = orphan_payments.count()

check(
    "Every payment references a valid ride",
    orphan_payment_count == 0,

    f"Orphan payments: {orphan_payment_count}"
)

[PASS] Every payment references a valid ride


In [0]:
def get_pipeline_counts():

    return {
        "bronze_drivers": spark.table(
            BRONZE_DRIVERS
        ).count(),

        "bronze_rides": spark.table(
            BRONZE_RIDES
        ).count(),

        "bronze_payments": spark.table(
            BRONZE_PAYMENTS
        ).count(),

        "silver_drivers": spark.table(
            SILVER_DRIVERS
        ).count(),

        "silver_rides": spark.table(
            SILVER_RIDES
        ).count(),

        "silver_payments": spark.table(
            SILVER_PAYMENTS
        ).count()
    }

In [0]:
counts_before_rerun = get_pipeline_counts()

print(counts_before_rerun)

{'bronze_drivers': 50, 'bronze_rides': 200, 'bronze_payments': 200, 'silver_drivers': 50, 'silver_rides': 200, 'silver_payments': 200}
